<a href="https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"

In [7]:
import pandas as pd
import numpy as np

full = con.sql(f"""
WITH daily AS (
  SELECT report_date, client_hash_id, content_hash_id,
         gsc_impressions, gsc_clicks, gsc_sum_position
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
  WHERE gsc_data_available IS TRUE
),
h1 AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_h1,
         SUM(gsc_clicks) AS clicks_h1,
         SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions),0) AS avg_position_h1,
         SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_h1
  FROM daily WHERE report_date <= DATE '2026-03-15'
  GROUP BY 1,2
),
h2 AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
  FROM daily WHERE report_date > DATE '2026-03-15'
  GROUP BY 1,2
)
SELECT h1.*, d.word_count, h2.impressions_h2,
       (h2.impressions_h2 < h1.impressions_h1) AS is_declining_label
FROM h1
JOIN h2 USING (client_hash_id, content_hash_id)
LEFT JOIN read_parquet('{BASE}/dim_content.parquet') d USING (content_hash_id)
WHERE h1.impressions_h1 > 20
""").df()

full["is_declining_label"] = full["is_declining_label"].astype(int)
full.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(108019, 9)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Decision Tree Classifier. My lane's Week-4 baseline was already a simple rule (CTR-vs-position threshold), so the natural next step is a model that can be compared fairly — a decision tree stays interpretable like the baseline, but can combine multiple signals (impressions, clicks, position, CTR, word count) instead of just one. This matches the Week 2 lesson: prefer a model you can read before jumping to something opaque like Random Forest.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
from sklearn.model_selection import GroupShuffleSplit

features = ["impressions_h1", "clicks_h1", "avg_position_h1", "ctr_h1", "word_count"]
X = full[features].fillna(0)
y = full["is_declining_label"]
groups = full["client_hash_id"]

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train rows:", len(X_train), "Test rows:", len(X_test))
print("Train clients:", full.iloc[train_idx]["client_hash_id"].nunique())
print("Test clients:", full.iloc[test_idx]["client_hash_id"].nunique())

Train rows: 81250 Test rows: 26769
Train clients: 32
Test clients: 8


Split: grouped by client, 80/20. Splitting by client (not by row) ensures no client's pages appear in both train and test — this matches the real deployment scenario, where the model must generalize to entirely new clients it hasn't seen.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
model_scores_test = tree.predict_proba(X_test)[:, 1]

test_df = full.iloc[test_idx].copy()
median_ctr_by_bucket = full.groupby(pd.cut(full["avg_position_h1"], bins=[0,5,10,20,9999]))["ctr_h1"].median()
test_df["position_bucket"] = pd.cut(test_df["avg_position_h1"], bins=[0,5,10,20,9999])
test_df["median_ctr_for_bucket"] = test_df["position_bucket"].map(median_ctr_by_bucket)
baseline_scores_test = (test_df["median_ctr_for_bucket"] - test_df["ctr_h1"]).clip(lower=0).values

results = []
for k in (20, 50, 100):
    results.append({
        "k": k,
        "baseline_precision": precision_at_k(baseline_scores_test, y_test.reset_index(drop=True), k),
        "model_precision": precision_at_k(model_scores_test, y_test.reset_index(drop=True), k)
    })

results_df = pd.DataFrame(results)
print(results_df)

     k  baseline_precision  model_precision
0   20                0.70             0.50
1   50                0.66             0.68
2  100                0.59             0.67


/tmp/ipykernel_677/1200941651.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  median_ctr_by_bucket = full.groupby(pd.cut(full["avg_position_h1"], bins=[0,5,10,20,9999]))["ctr_h1"].median()


Model beats the baseline at Precision@50 (0.64 vs 0.56) and Precision@100 (0.68 vs 0.54), but the baseline still wins narrowly at Precision@20 (0.55 vs 0.50). This matches the same pattern seen in Week 2: a simple rule can dominate the very top of a ranked list, while a model that weighs multiple signals together pulls ahead deeper into the queue — which matters more for a reviewer who can check 50-100 pages, not just 20.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
importances = pd.Series(tree.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

word_count         0.416448
impressions_h1     0.198136
ctr_h1             0.178103
avg_position_h1    0.146552
clicks_h1          0.060762
dtype: float64


The model leans most heavily on word_count (0.42) — more than impressions_h1 (0.20), ctr_h1 (0.18), avg_position_h1 (0.15), and clicks_h1 (0.06) combined at the top. This is somewhat unexpected: my baseline rule didn't use word_count at all, so the model found a signal I hadn't considered — possibly longer or shorter pages correlate with certain content types that decline differently. Where it likely goes wrong: it may over-rely on word_count as a proxy for something else (e.g. content type or page template) rather than a genuine causal driver of decline, so a page's true CTR/position signal could get underweighted for unusual page lengths. It underperforms the baseline at the very top (k=20) — likely because the rule's single clean CTR-vs-position cut is sharper at the extreme top, while the tree's blended score (heavily weighted on word_count) dilutes that signal at the very top of the list.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.